## Example for extracting data for GPT prompting

### requires python >= 3.10

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import os
import configparser
import json
import time

In [2]:
import tiktoken
import openai 
from openai import AzureOpenAI
from gpt_cost_estimator import CostEstimator
import pydantic
from pydantic import Field
from typing import Literal, List, Dict
from enum import Enum
from pydantic import BaseModel
import outlines
from outlines.models.openai import OpenAI, OpenAIConfig
from pydantic import ValidationError

/home/kaire/anaconda3/envs/gpt4/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pd.set_option('display.max_colwidth', None)

# OSA I : Andmed

## V1: teha kui pole olemas csv faili andmetega, muidu V2

### andmetabelid

In [2]:
filename = "../drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = """SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM lines_class_info4
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


### võtta ainult n80 tsooni lõksud

In [5]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [6]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [7]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
19889,peatuma,,in,3.551257,n80,282,257.0,337.0,0.00000,973,83,1056,858,1914
19994,möllama,,in,3.921070,n80,196,177.0,261.0,0.00000,409,27,436,495,931
20637,leiduma,,in,3.086569,n80,518,415.0,1689.0,0.00000,1614,190,1804,4760,6564
20673,naasma,,el,3.210249,n80,287,236.0,222.0,0.00000,907,98,1005,439,1444
19859,süttima,,in,3.879146,n80,199,177.0,233.0,0.00000,515,35,550,661,1211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18469,nappima,,in,3.496426,n80,140,114.0,198.0,0.00006,316,28,344,382,726
18887,sõitma,edasi,el,5.741467,n80,55,53.0,26.0,0.00007,107,2,109,30,139
20015,teatama,,ill,5.235216,n80,54,51.0,82.0,0.00008,113,3,116,640,756
20194,ootama,,ill,3.882643,n80,108,93.0,132.0,0.00008,236,16,252,231,483


In [8]:
filtered_class.to_sql("lines_class_info4_n80", conn, if_exists="replace", index=False)

335

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [3]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n80 tabelis
# iga lõksu kohta max 500 näidet ja iga unikaalse lemma kohta 1 näide

query = """
SELECT head_id, form, lemma, verb, verb_compound, morph_case, sentence_id, sentence, timex_tag, ekilex_tag, ner_tag
FROM (
    SELECT t2.*,
           ROW_NUMBER() OVER (
               PARTITION BY t2.verb, t2.verb_compound, t2.morph_case, t2.lemma
               ORDER BY t2.lemma
           ) AS rn_lemma,
           ROW_NUMBER() OVER (
               PARTITION BY t2.verb, t2.verb_compound, t2.morph_case
               ORDER BY t2.lemma
           ) AS rn_limit
    FROM spatial_obl t2
    INNER JOIN lines_class_info4_n80 fc
        ON t2.verb = fc.verb
       AND t2.verb_compound = fc.verb_compound
       AND t2.morph_case = fc.morph_case
    WHERE t2.timex_tag IS NULL
) sub
WHERE rn_lemma = 1        -- ensures only one row per lemma
  AND rn_limit <= 500     -- ensures max 500 rows per verb+verb_comp+morph_case
ORDER BY verb, verb_compound, morph_case;


"""


spatial_obl_ex = pd.read_sql(query, conn)


In [4]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
0,23747396,17das,17.,ajama,,in,15208349,Mingi aeg ajasin täna 17das servus kõiki Great...,None,None,None
1,18645542,Aafrikas,Aafrika,ajama,,in,11648113,"Kas teadsite , et Brechti suu oli nagu mustade...",None,location,LOC
2,26620253,Afganistanis,Afganistan,ajama,,in,17346120,Afganistanis ja Iraagis musklid suureks ajanud...,None,location,LOC
3,25724171,Alžeerias,Alžeeria,ajama,,in,16719535,Alžeerias ajas mind nii naerma - ümberringi ai...,None,location,LOC
4,17515699,Ameerikas,Ameerika,ajama,,in,10936172,"Lõppude lõpuks , kui Ameerikas ajavad metalpop...",None,location,LOC
...,...,...,...,...,...,...,...,...,...,...,...
44231,21960802,väikelinnas,väikelinn,üürima,,in,13774030,Arnold üürib väikelinnas korterit .,None,location,None
44232,6229577,võõrastemajas,võõrastemaja,üürima,,in,3867933,Haritud keskealine mees meenutab üht oma kunag...,None,location,None
44233,9515834,äärelinnas,äärelinn,üürima,,in,5927429,Noormees üürib väikese korteri äärelinnas .,None,location,None
44234,2889349,ühiselamus,ühiselamu,üürima,,in,1815434,Üliõpilane Viljar ( 22 ) üürib tuba Mustamäel ...,None,location,None


In [6]:
#spatial_obl_ex[(spatial_obl_ex["verb"]=="sõitma") & (spatial_obl_ex["verb_compound"]=='edasi') & (spatial_obl_ex["morph_case"]=='el')]

In [14]:
spatial_obl_ex[(spatial_obl_ex["verb"]=="kaduma") & (spatial_obl_ex["verb_compound"]=='') & (spatial_obl_ex["morph_case"]=='ill')]

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
8549,22377726,7.60sse,7.60,kaduma,,ill,14054596,"Võib ju öelda , et kuld kadus kaugushüppe 7.60...",None,None,None
8550,1440057,Ameerikasse,Ameerika,kaduma,,ill,904039,Maire Aunaste jättis võlgu ostetud köögitehnik...,None,location,LOC
8551,387398,Eestisse,Eesti,kaduma,,ill,242149,Esimese sularahaautomaadi tõi 1994. aastal Ees...,None,location,LOC
8552,2506922,Elvasse,Elva,kaduma,,ill,1575968,Klaasseni hinnangul kaob tal häirekeskuse ülev...,None,location,LOC
8553,737269,Euroopasse,Euroopa,kaduma,,ill,469893,"Toivo Kurmet kadus "" kusagile Euroopasse "" , k...",None,location,LOC
...,...,...,...,...,...,...,...,...,...,...,...
8837,14694016,tagatubadesse,tagatuba,kaduma,,ill,9140142,Samal ajal tõmbab don Giovanni lava ja saali h...,None,None,None
8838,7231654,taharuumidesse,taharuum,kaduma,,ill,4497639,Lint kaob ametnikega taharuumidesse .,None,None,None
8839,61180,talveõhtusse,talveõhtu,kaduma,,ill,35423,Ja kirjanik vaatab armastaval pilgul järele tü...,None,time,None
8840,13378108,taresse,tare,kaduma,,ill,8348985,"No jah , mida need aborigeenid siin mögisevad ...",None,location,None


In [17]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [18]:
df.to_csv("n80_examples_large_v1.csv", encoding="utf-8", index = False, sep="|")

query = """
SELECT t2.*
FROM spatial_obl t2
WHERE t2.timex_tag IS NULL

-- Keep only verb+comp+case that exist in lines_class_info4_n80
AND EXISTS (
    SELECT 1
    FROM lines_class_info4_n80 fc
    WHERE fc.verb = t2.verb
      AND fc.verb_compound = t2.verb_compound
      AND fc.morph_case = t2.morph_case
)

-- Keep only the first row per lemma (unique lemma)
AND t2.rowid = (
    SELECT MIN(t3.rowid)
    FROM spatial_obl t3
    WHERE t3.lemma = t2.lemma
      AND t3.verb = t2.verb
      AND t3.verb_compound = t2.verb_compound
      AND t3.morph_case = t2.morph_case
      AND t3.timex_tag IS NULL
)

-- Limit to max 500 rows per verb+verb_comp+morph_case group
AND (
    SELECT COUNT(*)
    FROM spatial_obl t4
    WHERE t4.timex_tag IS NULL
      AND t4.verb = t2.verb
      AND t4.verb_compound = t2.verb_compound
      AND t4.morph_case = t2.morph_case
      AND EXISTS (
            SELECT 1
            FROM lines_class_info4_n80 fc2
            WHERE fc2.verb = t4.verb
              AND fc2.verb_compound = t4.verb_compound
              AND fc2.morph_case = t4.morph_case
      )
      AND t4.rowid <= t2.rowid
) <= 500
ORDER BY t2.verb, t2.verb_compound, t2.morph_case;

"""

tbl2 = pd.read_sql(query, conn)

In [ ]:
# uue pika promtiga ja põhjendustega peaks 580 lauset olema 160K tokenit batch size 10, umbkaudu 2.5eur
# kui sama promptiga tahta 10 eur siis peaks alla 2300 lause võtma -> 2000
# kui põhjendust ei küsiks siis saaks rohkem aga kui palju?

In [20]:
df2 = pd.read_csv("n80_examples_large_v1.csv", encoding="utf-8", sep="|")

In [21]:
df2

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
0,22222603,Globenisse,Globen,võtma,kaasa,ill,13959468,Kolmest korpulentsest naislauljast koosnev The...,NaN,NaN,ORG
1,19340058,plagiaadikahtluses,plagiaadikahtlus,kaevama,NaN,in,12081233,"Vähe sellest , et tema müüginumbreid raske lüü...",NaN,NaN,NaN
2,19817151,hilispronksajal,hilispronksaja,paiknema,NaN,ad,12378462,Otse Kaali peakraatri nõlval paiknes hilispron...,NaN,NaN,NaN
3,4426350,Eestis,Eesti,üürima,NaN,in,2759941,Kolm päeva enne röövi üürisid vargad saadud in...,NaN,location,LOC
4,14341056,eestlastele,eestlane,valguma,NaN,all,8921549,Ei mõtle siin mitte niivõrd 1940. aastal eestl...,NaN,alive,NaN
...,...,...,...,...,...,...,...,...,...,...,...
44231,1174253,raudteejaamas,raudteejaam,lõhkema,NaN,in,737030,Kolmapäeval kell 17.24 teatas noor tütarlaps p...,NaN,location,NaN
44232,9092162,Lenskist,Lenski,evakueerima,NaN,el,5668833,Ööl vastu reedet evakueerisid eriolukordade mi...,NaN,NaN,ORG
44233,9876813,paika,paik,lubama,NaN,adit,6153274,Üks Pärnu mees lubas pärast kaldale jõudmist m...,NaN,location,NaN
44234,7618097,kostüümis,kostüüm,käima,ringi,in,4742806,""" Väiksena käisin ise ka kostüümis ringi , kui...",NaN,NaN,NaN


In [23]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
63,kaduma,NaN,ill,293
27,hakkama,NaN,ill,282
49,jälgima,NaN,in,270
250,suunduma,NaN,el,269
11,avastama,NaN,el,269
...,...,...,...,...
121,lendama,edasi,ill,35
296,turustama,NaN,in,35
241,soetama,NaN,ill,34
83,kolima,tagasi,ill,34


In [19]:
counts = df.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts

,verb,verb_compound,morph_case,count
63,kaduma,,ill,293
27,hakkama,,ill,282
49,jälgima,,in,270
11,avastama,,el,269
249,suunduma,,el,269
...,...,...,...,...
122,lendama,edasi,ill,35
296,turustama,,in,35
84,kolima,tagasi,ill,34
241,soetama,,ill,34


In [24]:
conn.close()

## V2 andmed: kui on eelnevalt salvestatud csv siis lugeda sisse

In [4]:
df = pd.read_csv("n80_examples_large_v1.csv", encoding="utf-8",  sep="|")

In [5]:
len(df)

44236

In [6]:
# kui faili on laused salvestatud shufflitud olekus, siis võiks võtta lihtsalt esimesed 2000

spatial_obl_ex = df.iloc[:10000]
spatial_obl_ex = spatial_obl_ex.sample(frac=1)

In [7]:
spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN
8815,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN
5631,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN
6492,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN
1312,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8966,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN
3967,21044230,Texases,Texas,tulistama,NaN,in,13152394,"Cheney jahilembus pääses tänavu ka meediasse , kui ta Texases kogemata oma linnujahikaaslast tulistas .",NaN,location,LOC
1530,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN
8524,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN


# OSA II : GPT

## GPT jaoks vajalik

In [8]:
config = configparser.ConfigParser()
status = config.read('azure.ini') 
assert status == ['azure.ini']

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [9]:
def in2json(sisend):
    return json.dumps(sisend, ensure_ascii=False)

In [10]:
SYSTEM_PROMPT = """
You are a classification assistant.
Your task: Given a list of JSON objects, each with keys "l" (sentence) and "c" (phrase), classify whether "c" is a location in the context of the sentence.
Locations: names of locations, buildings and bussinesses (bankhouse, studio, club), physical objects and living things, areas that have a defined geographic location, abstract places.
Not locations: actions and events, living beings who are action takers, state of being, ordinance, causal regulation, adverb of time, constructions and stamp expressions.
Output requirements:
- Respond with an array of JSON objects, one per input item.
- The output array must be in the exact same order as the input items.
- The output object must have "yes" (location) or "no" (not location) for each input item
- Do NOT give a longer reason like in few_shots.
Rules:
- Strict JSON only.
- No commentary.
- No markdown.
- No merging of items — one output per input.
"""

FEW_SHOTS = [
            {
            "role": "user",
            "content": in2json({"l": "Me läksime Pariisi", "c": "Pariisi"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "city", "r": "Phrase is a city name and therefore location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.", "c": "õmbluskoolis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "building", "r": "Phrase is a buidlding but also a bussiness."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.", "c": "Põhjapoolusele"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "location", "r": "Phrase is an area with defined geographical location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ta tuli idast kõikide oma raamatutega.", "c": "idast"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "direction", "r": "Phrase is an area with defined geographical location in the context of this sentence."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Mees istus peale pikka päeva uuesti sadulasse.", "c": "sadulasse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "object", "r": "Phrase is an object that can be defines as location."})
            },
    
            {
            "role": "user",
            "content": in2json({"l": "Ta alustas tööd kell üheksa", "c": "kell üheksa"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "time", "r": "Phrase is time expression not location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Avo Mäeseppa süüdistati selles, et ta eelmise aasta sügisel varastas magava J.P. põuetaskust salaja raha koos rahakotiga.", "c": "põuetaskust"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The theft did not happen in põuetasku and therefore is not a location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park.", "c": "restoranis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The party is planned to be in a restaurant but the action of planning is not happening there."})
            },

            {
            "role": "user",
            "content": in2json({"l": "HP700 ei ulatu enam SpeedTouchi Wifi'sse.", "c": "Wifi'sse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The geographic location of the phrase acn't be determined."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Minnie käis Barbra teadmata isegi kleidiproovis.", "c": "kleidiproovis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "event", "r": "Phrase is an event."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Rüselejal käsisid sussid", "c": "Rüselejal"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "actor", "r": "Phrase refers to action taker."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Nüüd siis istun sitas.", "c": "sitas"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "state", "r": "Phrase refers to state of being."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Protest on mitmekesine ja teravaimalt avaldub see kirjanduses.", "c": "teravaimalt"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "ordinance", "r": "Phrase refers to ordinance and is not location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Tulbisibul on siinkohal platseebo ja selle hävitamisel kaovad ka sümptomid.", "c": "hävitamisel"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "regulation", "r": "The phrase is causal regulation."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Rahulepingu kehtivusest lähtus omariikluse taastamise käigus Ülemnõukogu.", "c": "kehtivusest"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "construction", "r": "The phrase refers to construction of stamp expression."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Korraldasime seminari TTÜs.", "c": "TTÜs"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "org", "r": "The phrase refers to organization."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Väga hästi varjab päikesekiiri näiteks markiis.", "c": "markiis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "object", "r": "The phrase refers to an object that is in the inessive case but is not the location of the action."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ingridi puhul läks hiljem täkkesse just see ütelus.", "c": "täkkesse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "stamp", "r": "The phrase is in illativa case but is a stamp expression."})
            },

]
 

## tokenite arvutuseks

In [ ]:
# example batch

In [13]:
batch_items = []

for i in range(len(spatial_obl_ex)):
    ex = spatial_obl_ex.iloc[i]
    batch_items.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    if i==10:
        break

In [14]:
batch_items

['{"l": "Sellepärast ei suudagi ma lõplikult hukka mõista neid sulisid , kes röövisid Tartust terve sularahaautomaadi .", "c": "Tartust"}',
 '{"l": "enamus uued ussid levivad nii , et käid suvaliste lehekylgede peale ja see purakas automaatselt ronib sulle kõvakettale ..", "c": "kõvakettale"}',
 '{"l": "Kaie lubas mind oma proovidesse .", "c": "proovidesse"}',
 '{"l": "Ei mõtle siin mitte niivõrd 1940. aastal eestlastele idast kaela valgunud tapmiste ja röövimiste laviini .", "c": "eestlastele"}',
 '{"l": "Õnnetuseks sai kolmest Rodolfost parim nädal enne esietendust stipendiumi Itaaliasse .", "c": "Itaaliasse"}',
 '{"l": "Kuberner Arnold Schwarzenegger külastas koos abikaasaga tuletõrjestaapi ja lendas kopteril leekide kohal .", "c": "tuletõrjestaapi"}',
 '{"l": "7. juunil väljus Lissabonist rong Euroopa riikide kirjanikega , mis sõidab läbi kogu maailmajao ja saabub 30. juunil Tallinna .", "c": "Lissabonist"}',
 '{"l": "Kolmapäeval ootab hollandlasi Lissabonis poolfinaal ning siis tu

In [15]:
user_payload = {
        "Instruction": (
            "First interpret the few-shot examples. "
            "Then process the list called 'batch'. "
            "Output a JSON array with one item per batch entry, in the same order."
        ),
        "few_shots": FEW_SHOTS,
        "batch": batch_items
    }

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload)}
]

# oletame et mudeli output on sama palju tokeneid kui batch items


### manuaalne umbkaudne sisend

In [16]:
try:
    enc = tiktoken.encoding_for_model("gpt-4o")
    print("kasutab gpt-4o")
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(enc.encode(text))

def count_message_tokens(messages):
    total = 0
    for m in messages:
        total += len(enc.encode(m["role"]))
        total += len(enc.encode(m["content"]))
    return total

kasutab gpt-4o


In [17]:

print("KOKKU tokeneid:", count_message_tokens(messages))
print("System prompt tokeneid:", len(enc.encode(SYSTEM_PROMPT)))
print("Few-shots tokeneid:", count_message_tokens(FEW_SHOTS))
print("väljund tokeneid:", len(enc.encode(" ". join(batch_items))))
print(count_message_tokens(messages)+len(enc.encode(" ". join(batch_items))))

KOKKU tokeneid: 2302
System prompt tokeneid: 215
Few-shots tokeneid: 1078
väljund tokeneid: 589
2891


In [18]:
2891*200

578200

### cost estimator

In [19]:
messages2 = messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload)},
    {"role": "assistant", "content":in2json(batch_items)}
]

In [20]:
@CostEstimator()
def query_openai(model, messages, **kwargs):
    args_to_remove = ['mock', 'completion_tokens']

    for arg in args_to_remove:
        if arg in kwargs:
            del kwargs[arg]

    return openai.ChatCompletion.create(
        model = model,
        messages = messages,
        **kwargs)


responses = []
i = 0
#for i in tqdm(range(0,1)):
response = query_openai(
  model="gpt-4o",
  messages = messages,
  temperature=0,
  mock=True,
  completion_tokens=1
)

responses.append({
      'input': i,
      'output': response["choices"][0]["message"]["content"]
    })

print() # Empty line to display the total sum

# Print the responses
#print(responses)

Cost: $0.0081 | Total: $0.0081


In [21]:
CostEstimator.get_total_cost(CostEstimator)

0.0080875

In [22]:
0.0080875*200 #mul eurodes -> peab paketi koodis muutma ise summat, siis saab eurodes

1.6175

In [182]:
CostEstimator.reset()

## pydantic

In [11]:
class ClassificationDict(BaseModel):
    a: Literal["yes", "no"]
    #results: List[Literal["yes", "no"]]

class ClassificationAnswer(BaseModel):
    form : dict

In [12]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [13]:
def classify_batch(my_batch):

    print("classify", len(my_batch))
    max_att = 1
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "Instruction": (
                "Analyse the few-shot examples. "
                "Then process the list called 'batch'. "
                "Output a JSON array with one item per batch entry, in the same order."
                "Output a JSON array of EXACTLY N items (same length as 'batch' list) in the same order. Do not add or remove items."
            ),
            "few_shots": FEW_SHOTS,
            "batch": in2json(my_batch)
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": in2json(user_payload)}
        ]

        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0, # absoluutselt min väljund 
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError(f"Väljundis ei ole õige arv vastuseid. Peaks olema {len(batch)} aga on {len(data)}.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            #print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")
    # isegi kui ei saanud kõike kätte siis saab pärast äkki käsitsi midagi juurde panna
    return response, raw_output


In [14]:
def explain_non_locations(
    batch: List[Dict[str, str]],
    yes_no_results: List[str],
    yes_subset_ratio: float = 0.0
) -> Dict[int, str]:

    # Determine which indices to explain
    no_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "no"]
    yes_indices = [i for i, r in enumerate(yes_no_results) if r["a"] == "yes"]

    # Diagnostic subset
    diag_count = int(len(yes_indices) * yes_subset_ratio)
    diag_indices = yes_indices[:diag_count]

    explain_indices = no_indices + diag_indices
    if not explain_indices:
        return None, None, None

    items_to_explain = [
        {
            "index": i,
            "l": json.loads(batch[i])["l"],
            "c": json.loads(batch[i])["c"],
            "classification": yes_no_results[i]
        }
        for i in explain_indices
    ]

    messages = [
        {"role": "system", 
         "content": ("Explain why each phrase 'c' was classified as location ('yes') or not location ('no') in sentence 'l'." 
                       "Give one sentence answer."
                        "You MUST return only a pure JSON object, with no markdown, no code fences. "
                        "The output must be a mapping: {index: explanation}. "
                        "Do not include ```json or any backticks. Do not include commentary.")
        },
        {"role": "user", "content": (
            """For EACH item without missing any, return a JSON object mapping index → explanation in this format '{"0": "explanation", "3": "explanation"}'.\n"""
            "Items:\n" + in2json(items_to_explain)
        )}
    ]

    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=messages,
    )

    raw = response.choices[0].message.content.strip()

    # ---- Pydantic validation ----
    try:
        ClassificationAnswer(form = json.loads(raw))
    except ValidationError as e:
        raise ValueError(f"Invalid JSON structure returned in explanations:\n{e}")

    return response, raw, explain_indices

In [15]:
#df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)
df = spatial_obl_ex

In [16]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN
8815,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN
5631,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN
6492,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN
1312,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
8966,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN
3967,21044230,Texases,Texas,tulistama,NaN,in,13152394,"Cheney jahilembus pääses tänavu ka meediasse , kui ta Texases kogemata oma linnujahikaaslast tulistas .",NaN,location,LOC
1530,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN
8524,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN


In [17]:
def chunks(lst, size=10):
    """Yield successive chunks of size N."""
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

In [18]:
results = []
results2 = []
responses = []
explanations = []
explanations_all = {}

used_tokens = 0

# kui palju lauseid gtp-le korraga anda
bs = 12
# kui suure osa võtta "yes" vastustest "why" küsimusse
yes_subset_ratio = 0.2
batch_start_index = 0

batch_cnt = 0

rows = df.to_dict(orient="records")
# kui tahta kõiki näiteid anda gpt-le
for df_batch in chunks(rows, size=bs):
    batch = []
    for ex in df_batch:
        batch.append(in2json({"l": ex["sentence"], "c": ex["form"]}))

    batch_cnt += 1
    # esmalt klassifitseeri
    response, result = classify_batch(batch)
    result_yesno = json.loads(result)
    results += result_yesno
    results2.append(result_yesno)
    responses.append(response)
    used_tokens += response.usage.total_tokens

    # võtab välja kõik batchis olnud "no" ja mõne "yes" ja küsib why
    expl_response, batch_explanations, answered_idx = explain_non_locations(
            batch=batch,
            yes_no_results=result_yesno,
            yes_subset_ratio=yes_subset_ratio
        )

    # mapping: vastused õige lause+fraasiga kokku
    if batch_explanations is not None:
        used_tokens += expl_response.usage.total_tokens
        
        explanations.append(json.loads(batch_explanations))
        
        # Map batch-local -> global indices
        for local_i, explanation in json.loads(batch_explanations).items():
            global_i = batch_start_index + int(local_i)
            explanations_all[global_i] = explanation

    batch_start_index += len(batch)

    if used_tokens >= 3200000:
        print(f"Tehtud on {batch_cnt} batchi ehk {batch_cnt*bs} lauset")
        break    
    #break



classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
Error: Väljundis ei ole õige arv vastuseid. Peaks olema 12 aga on 11.
Batch failed after 1 attempts.
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
classify 12
Error: Väljundis ei ole

In [19]:
used_tokens # 2713238 batch 12 10K lauset -> ~ 5.8eur

2713238

In [20]:
len(results)

9984

In [29]:
16*bs # puuduva klassifikatsiooniga lauset

192

In [65]:
faulty_batches = {}
faulty_answers = {}
num_full_batches = int(len(df)/bs)
partial_batches = False if num_full_batches*bs == len(df) else True

if len(results) == len(df):
    df["classification"] = [r["a"] for r in results]

    new_explanations = []
    for i in range(len(df)):
        if i in explanations_all.keys():
            new_explanations.append( explanations_all[i])
        else:
            new_explanations.append("")
    
        
    df["explanation"] = new_explanations 


else: # juhuks kui mudel ei anna õiget arvu vastuseid tagasi
    new_results = []
    new_explanations = []
    for b, (batchres, expl) in enumerate(zip(results2, explanations),start=0):
        expected_len = bs if b < num_full_batches else len(df)-(num_full_batches*bs)
        
        if len(batchres) != expected_len and b < num_full_batches: # pole poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(expected_len)]
            #print(num_full_batches, b, len(replacement))
            new_results += replacement
            new_explanations += replacement
            faulty_batches[b] = batchres
            faulty_answers[b] = expl
        elif len(batchres) != expected_len and b >= num_full_batches and partial_batches:  # on poolikud batchid ja on puuduvaid vastuseid
            replacement = ["?" for i in range(len(expected_len))]
            new_results += replacement
            faulty_batches[b] = batchres
            new_explanations += replacement
            faulty_answers[b] = expl
        elif len(batchres) == expected_len: # kõik ok 
            new_results += [r["a"] for r in batchres]
            for i in range(len(batchres)):
                if str(i) in expl.keys():
                    new_explanations.append( expl[str(i)])
                else:
                    new_explanations.append("")
            
    
    df["classification"] = new_results
    df["explanation"] = new_explanations

    #df["explanation"] = new_explanations 

In [64]:
df

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
8815,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,
5631,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN,yes,
6492,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
1312,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8966,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN,yes,
3967,21044230,Texases,Texas,tulistama,NaN,in,13152394,"Cheney jahilembus pääses tänavu ka meediasse , kui ta Texases kogemata oma linnujahikaaslast tulistas .",NaN,location,LOC,yes,
1530,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
8524,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."


In [66]:
df[(df["explanation"]!="?") & (df["explanation"]!="")]

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
6492,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
2619,8111391,That's,That,lindistama,NaN,in,5054332,"Elvis , Scotty Moore ja Bill Black lindistasid "" That's all right "" -nimelise laulu Sun Recordsile 5. juulil 1954. aastal .",NaN,NaN,LOC,no,"The term 'That's' refers to part of a song title and not a location, so it is classified as no."
2094,3862261,ajakirjandusest,ajakirjandus,kostma,NaN,el,2408977,"Nüüd , kui esimesed kired Eurovisiooni lauluvõistluse järel on vaibumas , kostab mitmete riikide ajakirjandusest hääli , et Melodi Grand Prix vöistlusreegleid tuleks muutma hakata .",NaN,NaN,NaN,no,"The term 'ajakirjandusest' refers to journalism or media and not a physical geographic location, hence it is classified as no."
7182,56138,haiglasse,haigla,jooksma,NaN,ill,32675,"Kui ühel vendadest oli ninaoperatsioon , siis jooksis kogu suguvõsa haiglasse voodiservale tema kätt hoidma .",NaN,location,NaN,yes,"The phrase 'haiglasse' is classified as a location because it refers to a physical place, namely, a hospital."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4979,11945869,Valgamaalases,Valgamaalane,tutvustama,NaN,in,7440152,"Teost tutvustas 12. augusti Valgamaalases trükise toimetaja Hans Salm , valmimislugu on välja pandud keskraamatukogu teenindussaalis .",NaN,NaN,NaN,no,"The phrase 'Valgamaalases' refers to a publication and not a physical location, so it was classified as 'no'."
6988,12421351,kultuurist,kultuur,pääsema,välja,el,7748164,"Mida teha , et pääseksime Eestis välja sellisestpoliitilisest kultuurist , kus leiavad aset sellised juhtumid nagu viimane näide peaministriga ?",NaN,NaN,NaN,no,"The phrase 'kultuurist' pertains to cultural elements and not a specific location, so it was classified as 'no'."
1530,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
8524,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."


#### võtta vastused välja ja kontrollida, mida tabelisse panna

answers = []
shorts = []
long = []

#### kui mõnes batchis on rohkem/vähem vastuseid siis läheb siia listi
problematic = []

for elem in results:
    try:
        data = json.loads(elem)
        if len(data) != bs:
            for i in range(bs):
                answers.append("?")
                shorts.append("?")
                long.append("?")
            problematic.append(data)
        else:
            for item in data:
                answers.append(item["a"])
                shorts.append(item["s"])
                long.append(item["r"])
    except Exception as e:
        print(elem)

df["gpt_is_loc"] = answers
df["short_answ"] = shorts
df["long_answ"] = long

In [67]:
saving_fname = "n80_examples_large_v1_gpt_v1_10K_b12_v1.csv"

In [69]:
df.to_csv(saving_fname, encoding="utf-8", index = False, sep="|")

In [72]:
fname1 = saving_fname[:-4] + "_faulty_batches.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(faulty_batches, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_faulty_answers.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(faulty_answers, f, ensure_ascii=False)

fname1 = saving_fname[:-4] + "_yesno.json"
with open(fname1, "w", encoding="utf-8") as f:
    json.dump(results2, f, ensure_ascii=False)

fname2 = saving_fname[:-4] + "_why_reponses.json"
with open(fname2, "w", encoding="utf-8") as f:
    json.dump(explanations, f, ensure_ascii=False)

In [73]:
df[df["classification"]=="yes"] # 6474

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
8815,18564119,põõsas,põõsas,kükitama,NaN,in,11594114,"Kõik need praegused tähtsad tegelased kükitasid põõsas ja ootasid aega , et istuda toolile , mille teised olid kätte võidelnud .",NaN,NaN,NaN,yes,
5631,6712022,vitriinides,vitriin,ilutsema,NaN,in,4170566,"Kunstnike fantaasia võtab silme eest kirjuks - vitriinides ilutsevad rohelised , sinised , kollased , lillelised , liblikamustriga jm.",NaN,NaN,NaN,yes,
1312,9552490,töökohta,töökoht,pöörduma,tagasi,adit,5950379,"Hamburgis tehtud uuringu põhjal selgus , et ligi 70 protsenti naistest pöörduks pärast esimese lapse sündi heameelega vanasse töökohta tagasi , tegelikkuses teeb seda aga vaid 58 protsenti värsketest emadest .",NaN,NaN,NaN,yes,
5912,3305932,peatusesse,peatus,ootama,NaN,ill,2072089,"Damo ise võrdleb seda rongisõiduga : "" ma pole huvitatud juba möödunud maastike taasnägemisest / : / ootan järgmisesse peatusesse jõudmist , kusjuures eriti lõbus oleks veel jõuda peatusesse , mida pole kaardile märgitud "" .",NaN,NaN,NaN,yes,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8790,3866507,prügilasse,prügila,leidma,NaN,ill,2411565,"Ilma teejuhita prügilasse rada ei leia , sest teel puuduvad igasugused viidad .",NaN,location,NaN,yes,
6269,13953644,kruusaaugust,kruusaauk,saama,välja,el,8698768,Bronka saab kruusaaugust välja ja liipab edasi .,NaN,location,NaN,yes,
3920,16644781,rajakattel,rajakate,lamama,NaN,ad,10374293,Eesti parim sportlane Erki Nool lamab Kadrioru staadioni väsinud rajakattel .,NaN,location,NaN,yes,
8966,10441673,Kumusse,kumu,hakkama,NaN,ill,6492455,"Näiteks tunnelit , mille kaudu Kumusse veoautod väärtuslike maalikoormatega sõitma hakkavad , polnud üldse alguses plaanis ehitadagi .",NaN,NaN,NaN,yes,


In [75]:
df[(df["classification"]=="yes") & (df["explanation"]!="")]

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
989,12979894,autodesse,auto,müüma,NaN,ill,8089054,“ Nad tassivad raskeid mitmekiloseid pakke ja müüvad lehti otse keset liiklust autodesse .,NaN,NaN,NaN,yes,"The term 'autodesse' refers to cars, which are considered a physical location, so it is classified as yes."
7182,56138,haiglasse,haigla,jooksma,NaN,ill,32675,"Kui ühel vendadest oli ninaoperatsioon , siis jooksis kogu suguvõsa haiglasse voodiservale tema kätt hoidma .",NaN,location,NaN,yes,"The phrase 'haiglasse' is classified as a location because it refers to a physical place, namely, a hospital."
1721,3855195,Kenemast,Kenema,rändama,NaN,el,2404912,"Kenemast rändavad kivid pealinna Freetowni Liibanonist pärit äripartnerile , kes viib teemandid riigist välja .",NaN,NaN,NaN,yes,The phrase 'Kenemast' refers to a specific location (the town of Kenema in Sierra Leone) and is therefore classified as a location ('yes').
6896,4643907,Kuubasse,Kuuba,maksma,NaN,ill,2896397,"Tõsi , "" Päikesepüüdja "" on samm edasi eelmisel telehooajal hommikuti eetris olnud Anneli Järveti ja Domina koostöös sündinud reisiülevaatest saates "" Kaunimaks kõikjal "" , kus oli selgelt näha , kes maksis võttegrupi kohalesõidukulud Kuubasse või kuhu iganes .",NaN,location,LOC,yes,"The phrase 'Kuubasse' refers to a specific geographical location (Cuba), so it was classified as 'yes'."
8848,21260277,koolis,kool,juhatama,NaN,in,13288680,"Vahel tahavad tööandjad teada nende õppejõudude nimesid , kes kõnealuses koolis teaduskondi ja õppetoole juhatavad .",NaN,NaN,NaN,yes,"The phrase 'koolis' was classified as 'yes' because it specifies a physical place of learning, which qualifies as a location."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8556,12639641,nõmmemetsades,nõmmemets,hulkuma,NaN,in,7895084,Siis hulkusid nad nõmmemetsades .,NaN,location,NaN,yes,"The phrase 'nõmmemetsades' refers to 'moor forests,' which is a type of location or landscape."
1805,7699947,tallu,talu,lubama,NaN,adit,4794600,"Eesti Energia autod keeravad metsateele , aga Nõmme tallu energia-mehed kiiresti elektrit ei luba , see tuleb teist kaudu vedada kui seni .",NaN,NaN,NaN,yes,"The phrase 'tallu' refers to a 'farmstead,' which is a specific type of location or place."
6926,4828792,Pärnus,Pärnu,nappima,NaN,in,3010384,""" Need numbrid näitavad ilmekalt , et Pärnus napib elamukrunte , "" selgitas kinnisvarabüroo LVM juhatuse liige Ingmar Saksing .",NaN,location,LOC,yes,"The term 'Pärnus' refers to a specific geographical place, Pärnu, so it is classified as 'yes'."
3667,1655206,Portugalist,Portugal,lendama,NaN,el,1040882,"Nädalasel Euroopa-ringreisil viibiv USA president Bill Clinton lendas eile Portugalist Saksamaale , kust edasi viib reis ta homseks Moskvasse kohtumisele Vene presidendi Vladimir Putiniga .",NaN,location,LOC,yes,"The phrase 'Portugalist' refers to a specific geographic location, Portugal, and was classified as 'yes'."


In [74]:
df[df["classification"]=="no"] # 3334

,head_id,form,lemma,verb,verb_compound,morph_case,sentence_id,sentence,timex_tag,ekilex_tag,ner_tag,classification,explanation
6492,15715,MTVs,MTV,ringlema,NaN,in,9160,"Kolm Sahlene soolosinglit on ringelnud MTVs , jõudes MTV Nordic Top 5 hulka .",NaN,NaN,NaN,no,"The term 'MTVs' refers to a TV network brand and not a geographic place, so it is not classified as a location."
2619,8111391,That's,That,lindistama,NaN,in,5054332,"Elvis , Scotty Moore ja Bill Black lindistasid "" That's all right "" -nimelise laulu Sun Recordsile 5. juulil 1954. aastal .",NaN,NaN,LOC,no,"The term 'That's' refers to part of a song title and not a location, so it is classified as no."
2094,3862261,ajakirjandusest,ajakirjandus,kostma,NaN,el,2408977,"Nüüd , kui esimesed kired Eurovisiooni lauluvõistluse järel on vaibumas , kostab mitmete riikide ajakirjandusest hääli , et Melodi Grand Prix vöistlusreegleid tuleks muutma hakata .",NaN,NaN,NaN,no,"The term 'ajakirjandusest' refers to journalism or media and not a physical geographic location, hence it is classified as no."
62,12418343,Liidus,liit,jooma,NaN,in,7746276,"Nõukogude Eestis elas üks nn raamaturahvas , kes seda maad külastanud türgi luuletaja Nazõm Hikmeti sõnul “ luges kõige rohkem luulet ja jõi kõige rohkem viina ” terves Nõukogude Liidus .",NaN,NaN,LOC,no,"The phrase 'Liidus' is not classified as a location because it is part of the name 'Nõukogude Liidus', which refers to an organization (Soviet Union) rather than a specific geographical location."
5325,27385004,kaugõppesse,kaugõpe,saama,sisse,ill,17996437,"Jõgeva elanik , kahe lapse ema Katri ( 32 ) sai sisse Tallinna Pedagoogikaülikooli alghariduspedagoogika kaugõppesse .",NaN,NaN,NaN,no,The phrase 'kaugõppesse' is not classified as a location because it refers to a mode of education (distance learning) rather than a physical place.
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4979,11945869,Valgamaalases,Valgamaalane,tutvustama,NaN,in,7440152,"Teost tutvustas 12. augusti Valgamaalases trükise toimetaja Hans Salm , valmimislugu on välja pandud keskraamatukogu teenindussaalis .",NaN,NaN,NaN,no,"The phrase 'Valgamaalases' refers to a publication and not a physical location, so it was classified as 'no'."
6988,12421351,kultuurist,kultuur,pääsema,välja,el,7748164,"Mida teha , et pääseksime Eestis välja sellisestpoliitilisest kultuurist , kus leiavad aset sellised juhtumid nagu viimane näide peaministriga ?",NaN,NaN,NaN,no,"The phrase 'kultuurist' pertains to cultural elements and not a specific location, so it was classified as 'no'."
1530,26748522,Liitu,liit,pagema,NaN,adit,17445921,"Kui riik seda ei teeks , toimuks paari põlvkonna jooksul perifeerias , näiteks , Piirissaarel ja Ruhnu saarel , Alutaguse metsades ja mujal migratsioon , tühjenemine ning Euroopa Liit võiks teha 21. sajandi alguseks Eestile ettepaneku asustada nendele tühjenenud maadele Euroopa Liitu varju pagenud pagulasi kolmandast maailmast , mis tekitaks Eestile tõenäoliselt sama keerulise probleemi kui hetkel riigi ääremaade rahaline abistamine .",NaN,NaN,NaN,no,"The phrase 'Liitu' refers to 'European Union', which is an organization and not a location, hence classified as 'no'."
8524,1772660,meediasse,meedia,hakkama,NaN,ill,1114969,"Ka eesti meediasse on hakanud imbuma uudiseid ja usutlusi memeetikast - viimaste aastakümnetega esiletõusnud teadusharust , mis uurib ideede evolutsiooni ja mõjulepääsmist teadvuses ja kultuurides .",NaN,NaN,NaN,no,"The phrase 'meediasse' translates to 'into the media', which is a concept and not a specific location, hence classified as 'no'."
